In [2]:
# -*- coding: utf-8 -*-
"""
Refresh extractor for UNDERCOUNT repos — pinned to cutoff commit (Aug 10, 2025 UTC).

What’s new vs your original:
- Enforces cutoff by checking out the default-branch commit at/before 2025-08-10 23:59:59 +0000
- Uses partial clone (--filter=blob:none, --no-checkout) and then checks out the cutoff SHA
- Builds html_url with the pinned commit (not the branch) for stable references
- Writes everything into: Undercounts Refereshing

Kept from your original:
- CI YAML pattern logic (unchanged)
- Buckets & flat file naming scheme
- Two buckets: All_Config_Files and All_Test_Files
"""

from __future__ import annotations

import csv
import hashlib
import os
import re
import shutil
import stat
import subprocess
from pathlib import Path
from typing import Optional, Set
from urllib.parse import urlparse

import pandas as pd

# ========= CUT-OFF =========
CUTOFF_ISO = "2025-08-10 23:59:59 +0000"

# ========= PATHS (UNDERCOUNTS REFRESH) =========
base_dir = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Undercounts Refereshing")
csv_path = base_dir / "URL_List.csv"    # <- put the 8 undercount repo URLs here (column 'github_url')

clone_dir = base_dir / "Cloned_At_Cutoff"
config_bucket = base_dir / "All_Config_Files"
tests_bucket = base_dir / "All_Test_Files"

flat_index_csv = base_dir / "All_Config_Index.csv"  # unified index for saved files

for d in [clone_dir, config_bucket, tests_bucket]:
    d.mkdir(parents=True, exist_ok=True)

INDEX_FIELDS = [
    "owner", "repo", "repo_url", "default_branch", "commit_sha",
    "relative_path", "filename", "flat_filename", "ci_platform",
    "html_url", "saved_to", "bucket", "components", "ci_root"
]

# ========= CI patterns (UNCHANGED) =========
ci_patterns = {
    r'\.travis\.yml$': 'Travis_CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'Circle_CI',
    r'\.circleci/config\.yml$': 'Circle_CI',
    r'azure-pipelines\.yml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}

ci_provider_token = {
    "GitHub_Actions": "github_actions",
    "GitLab": "gitlab",
    "Circle_CI": "circle_ci",
    "Azure_Pipelines": "azure_pipelines",
    "Travis_CI": "travis_ci",
    "Bitrise": "bitrise",
    "Bitbucket": "bitbucket",
    "Jenkins": "jenkins",
    "Bamboo": "bamboo",
    "Codeship": "codeship",
    "GoCD": "gocd",
    "Cirrus": "cirrus",
    "Wercker": "wercker",
    "Semaphore": "semaphore",
    "Nevercode": "codemagic",
    "AppVeyor": "appveyor",
    "Other": "other",
}

# ========= Allowlisted CI JSON filenames =========
LIKELY_CI_JSON: set[str] = {
    "android-studio-loading.json",
    "saucectl.config.json",
    "firebase.json",
    "test-lab.json",
}

# ========= Helpers =========
def run(cmd: list[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=check)

def sanitize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9._+-]", "_", s.lower())

def split_stem_and_suffixes(name: str) -> tuple[str, str]:
    p = Path(name)
    suffixes = ''.join(p.suffixes)
    if suffixes:
        return name[:-len(suffixes)], suffixes
    return name, ""

def _counter_candidate(base_name: str, n: int) -> str:
    if n == 1:
        return base_name
    if "++" in base_name:
        head, tail = base_name.split("++", 1)
        tail_stem, tail_suf = split_stem_and_suffixes(tail)
        return f"{head}++{tail_stem}__{n}{tail_suf}"
    stem, suf = split_stem_and_suffixes(base_name)
    return f"{stem}__{n}{suf}"

def resolve_counter_name(bucket_dir: Path, base_name: str, in_memory_taken: Set[str]) -> str:
    n = 1
    while True:
        candidate = _counter_candidate(base_name, n)
        if candidate not in in_memory_taken and not (bucket_dir / candidate).exists():
            in_memory_taken.add(candidate)
            return candidate
        n += 1

def file_sha1_hex(path: Path, chunk_size: int = 1 << 20) -> str:
    import hashlib
    h = hashlib.sha1()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def make_flat_filename(owner: str, project: str, ci_platform: str, file_name: str, taken: Set[str]) -> str:
    owner_tok = sanitize_token(owner)
    project_tok = sanitize_token(project)
    ci_tok = sanitize_token(ci_platform or "other")
    file_lower = file_name.lower()
    return f"{owner_tok}.{project_tok}__{ci_tok}++{file_lower}"

def save_file(bucket_dir: Path, flat_filename: str, src: Path, in_memory_taken: Set[str]) -> tuple[Path, str]:
    bucket_dir.mkdir(parents=True, exist_ok=True)
    final_name = resolve_counter_name(bucket_dir, flat_filename, in_memory_taken)
    dest = bucket_dir / final_name
    content_sha1 = file_sha1_hex(src)
    shutil.copy2(src, dest)
    return dest, content_sha1

# ---- Classification (path/name only; content used for Manifest <activity> check) ----
SHELL_EXTS: set[str] = {".sh", ".bash", ".zsh", ".ksh", ".bat", ".cmd", ".ps1", ".psm1", ".psd1"}
MAKEFILES: set[str] = {"makefile", "gnumakefile", "makefile.win", "makefile.mak"}

def is_shell_or_make(file_path: Path) -> tuple[bool, str]:
    name = file_path.name.lower()
    ext = file_path.suffix.lower()
    if name in MAKEFILES:
        return True, "shell"
    if ext in SHELL_EXTS:
        if ext in {".ps1", ".psm1", ".psd1"}:
            return True, "shell_ps"
        if ext in {".bat", ".cmd"}:
            return True, "shell_win"
        return True, "shell"
    return False, ""

def is_gradle_or_settings(file_path: Path) -> tuple[bool, str]:
    n = file_path.name.lower()
    if n.endswith(".gradle") or n.endswith(".gradle.kts"):
        return True, "gradle"
    if n in {"gradle.properties", "settings.gradle", "settings.gradle.kts"}:
        return True, "gradle"
    return False, ""

def is_manifest(file_path: Path) -> bool:
    return file_path.name.lower() == "androidmanifest.xml"

def manifest_has_activity(file_path: Path) -> bool:
    try:
        txt = file_path.read_text(encoding="utf-8", errors="ignore")
        return re.search(r"<\s*activity\b", txt, re.IGNORECASE) is not None
    except Exception:
        return False

def is_allowlisted_ci_json(file_path: Path) -> tuple[bool, str]:
    n = file_path.name.lower()
    if n in LIKELY_CI_JSON:
        if n.startswith("saucectl"):
            return True, "sauce_labs"
        if n.startswith("android-studio"):
            return True, "android_studio"
        if n in {"firebase.json", "test-lab.json"}:
            return True, "firebase_test_lab"
        return True, "ci_json"
    return False, ""

def is_flutter_pubspec(file_path: Path) -> bool:
    return file_path.name.lower() == "pubspec.yaml"

def is_flutter_test(file_path: Path) -> tuple[bool, str]:
    rel = file_path.as_posix().lower()
    if rel.startswith("integration_test/") or "/integration_test/" in rel:
        return True, "androidtest_dart"
    if rel.startswith("test_driver/") or "/test_driver/" in rel:
        return True, "androidtest_dart"
    return False, ""

def is_androidtest_code(file_path: Path) -> tuple[bool, str]:
    rel = file_path.as_posix().lower()
    if "/src/" in rel and "androidtest" in rel:
        if file_path.suffix.lower() == ".kt":
            return True, "androidtest_kotlin"
        if file_path.suffix.lower() == ".java":
            return True, "androidtest_java"
    return False, ""

# ========= Read URL list =========
if not csv_path.exists():
    raise FileNotFoundError(f"URL list not found: {csv_path}")

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
if "github_url" not in df.columns:
    raise KeyError("URL_List.csv must have a 'github_url' column.")
df = df[df["github_url"].astype(str).str.startswith("https://")].copy()

# ========= Prepare flat index =========
if not flat_index_csv.exists():
    with open(flat_index_csv, "w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=INDEX_FIELDS).writeheader()

# ========= Process each repo =========
for idx, row in df.reset_index(drop=True).iterrows():
    url = str(row["github_url"]).strip()
    owner_repo = urlparse(url).path.strip("/").split("/")
    if len(owner_repo) < 2:
        print(f"⚠️ Skipping invalid URL: {url}")
        continue

    owner, project = owner_repo[0], owner_repo[1].replace(".git", "")
    repo_name = f"{owner}.{project}"
    repo_path = clone_dir / repo_name

    print(f"\n🔧 [{idx+1}/{len(df)}] {repo_name} — pinning to cutoff …")

    # --- Determine default branch ---
    default_branch = None
    try:
        out = run(["git", "ls-remote", "--symref", url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]
                if ref.startswith("refs/heads/"):
                    default_branch = ref.split("/", 2)[2]
                    break
    except Exception:
        pass
    if default_branch is None:
        for guess in ("main", "master"):
            cp = run(["git", "ls-remote", url, f"refs/heads/{guess}"], check=False)
            if cp.returncode == 0 and cp.stdout.strip():
                default_branch = guess
                break
    if default_branch is None:
        print(f"❌ Could not determine default branch for {url}; skipping.")
        continue

    # --- Clone (partial) and find cutoff commit ---
    if repo_path.exists():
        shutil.rmtree(repo_path, ignore_errors=True)
    try:
        # Partial clone, no checkout yet
        run(["git", "clone", "--filter=blob:none", "--no-checkout",
             "--single-branch", "--branch", default_branch, url, str(repo_path)])
    except subprocess.CalledProcessError as e:
        print(f"❌ Clone failed for {repo_name}: {e.stdout.strip()}")
        continue

    # cutoff SHA on default branch
    try:
        cp = run(
            ["git", "-C", str(repo_path), "rev-list", "-n", "1", "--first-parent",
             f"--before={CUTOFF_ISO}", f"origin/{default_branch}"],
            check=False
        )
        cutoff_sha = (cp.stdout or "").strip()
    except Exception as e:
        cutoff_sha = ""

    if not cutoff_sha:
        print(f"⚠️ No commit on {default_branch} before {CUTOFF_ISO} for {repo_name}; skipping.")
        shutil.rmtree(repo_path, ignore_errors=True)
        continue

    # checkout the cutoff commit (blobs will be fetched on-demand due to partial clone)
    try:
        run(["git", "-C", str(repo_path), "checkout", "--quiet", cutoff_sha])
    except subprocess.CalledProcessError as e:
        print(f"❌ Checkout failed for {repo_name} at {cutoff_sha}: {e.stdout.strip()}")
        shutil.rmtree(repo_path, ignore_errors=True)
        continue

    # --- Walk tree & save files into two buckets ---
    any_yml = False
    used_names_config: Set[str] = set()
    used_names_tests: Set[str] = set()
    seen_realpaths: Set[Path] = set()

    for root, _, files in os.walk(repo_path):
        for file in files:
            file_path = Path(root) / file
            try:
                realp = file_path.resolve()
            except Exception:
                realp = file_path
            if realp in seen_realpaths:
                continue
            seen_realpaths.add(realp)

            rel_path = str(file_path.relative_to(repo_path)).replace("\\", "/")
            filename_lower = file.lower()
            dest_path = None
            saved_name = None
            ci_platform = "Other"
            bucket = ""
            ci_root_value = ""

            try:
                # 1) YAML (UNCHANGED LOGIC)
                if filename_lower.endswith(('.yml', '.yaml')):
                    matched_ci_type = None
                    for pattern, platform in ci_patterns.items():
                        if re.search(pattern, rel_path, re.IGNORECASE):
                            matched_ci_type = platform
                            break
                    if matched_ci_type:
                        ci_platform = ci_provider_token.get(matched_ci_type, "other")
                        flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                        dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                        saved_name = dest_path.name
                        bucket = "All_Config_Files"
                        any_yml = True
                        ci_root_value = "yes"
                    else:
                        continue

                # 2) Shell & Makefiles (CONFIG)
                elif is_shell_or_make(file_path)[0]:
                    ci_platform = is_shell_or_make(file_path)[1]
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                    dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                    saved_name = dest_path.name
                    bucket = "All_Config_Files"

                # 3) Gradle & settings (CONFIG)
                elif is_gradle_or_settings(file_path)[0]:
                    ci_platform = "gradle"
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                    dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                    saved_name = dest_path.name
                    bucket = "All_Config_Files"

                # 4) AndroidManifest with <activity> (CONFIG)
                elif is_manifest(file_path) and manifest_has_activity(file_path):
                    ci_platform = "manifest_activity"
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                    dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                    saved_name = dest_path.name
                    bucket = "All_Config_Files"

                # 5) Allowlisted CI JSON (CONFIG)
                elif is_allowlisted_ci_json(file_path)[0]:
                    ci_platform = is_allowlisted_ci_json(file_path)[1]
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                    dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                    saved_name = dest_path.name
                    bucket = "All_Config_Files"

                # 6) Flutter pubspec.yaml (CONFIG)
                elif is_flutter_pubspec(file_path):
                    ci_platform = "flutter"
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_config)
                    dest_path, content_sha1 = save_file(config_bucket, flat_name, file_path, used_names_config)
                    saved_name = dest_path.name
                    bucket = "All_Config_Files"

                # 7) Flutter integration tests (TESTS)
                elif is_flutter_test(file_path)[0]:
                    ci_platform = "androidtest_dart"
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_tests)
                    dest_path, content_sha1 = save_file(tests_bucket, flat_name, file_path, used_names_tests)
                    saved_name = dest_path.name
                    bucket = "All_Test_Files"

                # 8) AndroidTest code .kt/.java (TESTS)
                elif is_androidtest_code(file_path)[0]:
                    ci_platform = is_androidtest_code(file_path)[1]
                    flat_name = make_flat_filename(owner, project, ci_platform, filename_lower, used_names_tests)
                    dest_path, content_sha1 = save_file(tests_bucket, flat_name, file_path, used_names_tests)
                    saved_name = dest_path.name
                    bucket = "All_Test_Files"
                else:
                    continue

                # === Write unified CSV index (pinned commit URL) ===
                with open(flat_index_csv, "a", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(f, fieldnames=INDEX_FIELDS)
                    writer.writerow({
                        "owner": owner,
                        "repo": project,
                        "repo_url": url.strip(),
                        "default_branch": default_branch,
                        "commit_sha": cutoff_sha,
                        "relative_path": rel_path,
                        "filename": file,
                        "flat_filename": saved_name,
                        "ci_platform": ci_platform,
                        "html_url": f"https://github.com/{owner}/{project}/blob/{cutoff_sha}/{rel_path}",
                        "saved_to": str(dest_path),
                        "bucket": bucket,
                        "components": "",
                        "ci_root": ci_root_value,
                    })

            except Exception as e:
                print(f"⚠️ Could not process {rel_path} in {repo_name}: {e}")

    # Cleanup to keep only the extracted files (optional)
    try:
        def force_remove_readonly(func, path, _):
            os.chmod(path, stat.S_IWRITE)
            func(path)
        shutil.rmtree(repo_path, onerror=force_remove_readonly)
    except Exception as e:
        print(f"⚠️ Unable to delete working clone for {repo_name}: {e}")

print("\n✅ Refresh complete. Outputs saved under:")
print(f"  - {config_bucket}")
print(f"  - {tests_bucket}")
print(f"  - {flat_index_csv}")



🔧 [1/9] labexp.osmtracker-android — pinning to cutoff …

🔧 [2/9] segler-alex.RadioDroid — pinning to cutoff …

🔧 [3/9] vgaidarji.ci-matters — pinning to cutoff …

🔧 [4/9] renyuneyun.Easer — pinning to cutoff …

🔧 [5/9] TechbeeAT.jtxBoard — pinning to cutoff …

🔧 [6/9] kickstarter.android-oss — pinning to cutoff …

🔧 [7/9] spacecowboy.Feeder — pinning to cutoff …

🔧 [8/9] maxkeppeler.sheets-compose-dialogs — pinning to cutoff …

🔧 [9/9] twilio.twilio-voice-react-native — pinning to cutoff …

✅ Refresh complete. Outputs saved under:
  - C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Undercounts Refereshing\All_Config_Files
  - C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Undercounts Refereshing\All_Test_Files
  - C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Undercounts Refereshing\All_Config_Index.csv
